# 01 · Generate the Darcy-flow dataset (PDEBench setting, train / val / test)

Solves `-div(a ∇u) = 1` on the unit square in the **PDEBench 2D DarcyFlow (β = 1.0)** setting, so a
model trained on this data can later be trained / evaluated on PDEBench with the same settings:

| | this notebook | PDEBench `2D_DarcyFlow_beta1.0_Train.hdf5` |
|---|---|---|
| grid | 128×128 cell centres | 128×128 cell centres |
| a(x) | 5 Gaussian bumps → {1, 0.1} | 5 Gaussian bumps → {1, 0.1} |
| forcing | β = 1 | β = 1 |
| u(x) | exact steady solve (same finite-volume stencil) | time-stepped to t = 2 |
| file layout | `nu`, `tensor`, `x-coordinate`, `y-coordinate`, `beta` | same |

Self-contained: no repo clone. Each split has its own seed, so val and test are truly held out.

**Kaggle settings:** no GPU needed (the solver is CPU-only); Internet not needed.
Run the **ETA** cell first to see how long generation will take, then Run All / Save Version: the
`data/` folder becomes this notebook's output, which you attach as an input to `02_train_ddp.ipynb`.

In [ ]:
import os

# ============================== CONFIG ==============================
WORK_DIR   = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
GRID       = 128                                  # PDEBench resolution
SPLITS     = {"train": (9000, 0),                 # name: (n_samples, seed); 10000 total = PDEBench
              "val":   (500, 1),
              "test":  (500, 2)}
N_WORKERS  = os.cpu_count()                       # parallel solver processes
CHUNK_SIZE = 200                                  # samples per job (PDEBench batch); fixes the output
OUT_DIR    = f"{WORK_DIR}/data"
# ====================================================================
os.chdir(WORK_DIR)
print("work dir:", WORK_DIR, "| cpus:", os.cpu_count(), "| workers:", N_WORKERS)

In [ ]:
%%writefile gen_darcy.py
"""Darcy dataset generation in the PDEBench 2D DarcyFlow setting (train / val / test).

Written by notebooks/01_generate_dataset.ipynb. Self-contained (numpy + scipy + h5py),
no repo clone needed.

    python gen_darcy.py --splits train:9000:0,val:500:1,test:500:2 --grid 128 \
        --workers 4 --out-dir data

a(x) is sampled per chunk (cheap, fixes the output); the expensive solves are spread over
the workers in small blocks, so all cores stay busy and progress / ETA update smoothly.

Setting (matches PDEBench 2D_DarcyFlow_beta1.0, Takamoto et al. 2022):
  -div(a grad u) = beta on (0,1)^2, beta = 1, u = 0 on the boundary
  grid: n x n cell centres (i + 0.5) / n
  a(x): sum of 5 Gaussian bumps exp(-|x - m|^2 / s), m ~ U[0,1]^2, s ~ 0.5 U[0,1],
        thresholded at the mean over the chunk (PDEBench: per batch of 200) to {1, 0.1}
  u(x): steady state solved directly (sparse LU) with PDEBench's finite-volume stencil
        (h = 1/n, arithmetic face averaging). PDEBench instead time-steps to t = 2.

Output per split: darcy_<split>.hdf5 in PDEBench's layout
  nu (N, n, n), tensor (N, 1, n, n), x-coordinate (n,), y-coordinate (n,), attrs beta.
Each chunk uses np.random.default_rng([seed, chunk_id]): identical output for any --workers.
"""

from __future__ import annotations

import argparse
import json
import os
import time
from multiprocessing import Pool
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")  # one solver per core, no oversubscription

import h5py  # noqa: E402
import numpy as np  # noqa: E402
from scipy import sparse  # noqa: E402
from scipy.sparse.linalg import spsolve  # noqa: E402

BETA, LO, HI, N_BUMPS = 1.0, 0.1, 1.0, 5


def cell_centres(n: int) -> np.ndarray:
    return (np.arange(n) + 0.5) / n


def sample_permeability(n_samples: int, n: int, rng: np.random.Generator) -> np.ndarray:
    """(n_samples, n, n) two-phase field in {LO, HI}; axis 1 is x, axis 2 is y."""
    xc = cell_centres(n)
    m = rng.uniform(0.0, 1.0, (n_samples, N_BUMPS, 2))
    s = np.maximum(0.5 * rng.uniform(0.0, 1.0, (n_samples, N_BUMPS)), 1e-6)
    field = np.zeros((n_samples, n, n))
    for k in range(N_BUMPS):
        dx2 = (xc[None, :] - m[:, k, 0:1]) ** 2   # (N, n)
        dy2 = (xc[None, :] - m[:, k, 1:2]) ** 2
        field += np.exp(-(dx2[:, :, None] + dy2[:, None, :]) / s[:, k, None, None])
    return np.where(field > field.mean(), HI, LO)


def solve(a: np.ndarray, f: float = BETA) -> np.ndarray:
    """Steady -div(a grad u) = f, u = 0 one cell outside the edge, h = 1/n, arithmetic faces."""
    n = a.shape[0]
    h = 1.0 / n
    a_e = np.empty((n, n)); a_w = np.empty((n, n)); a_n = np.empty((n, n)); a_s = np.empty((n, n))
    a_e[:, :-1] = 0.5 * (a[:, :-1] + a[:, 1:]); a_e[:, -1] = a[:, -1]
    a_w[:, 1:] = 0.5 * (a[:, 1:] + a[:, :-1]); a_w[:, 0] = a[:, 0]
    a_n[:-1, :] = 0.5 * (a[:-1, :] + a[1:, :]); a_n[-1, :] = a[-1, :]
    a_s[1:, :] = 0.5 * (a[1:, :] + a[:-1, :]); a_s[0, :] = a[0, :]
    idx = np.arange(n * n).reshape(n, n)
    rows = [idx.ravel(), idx[:, :-1].ravel(), idx[:, 1:].ravel(), idx[:-1, :].ravel(), idx[1:, :].ravel()]
    cols = [idx.ravel(), idx[:, 1:].ravel(), idx[:, :-1].ravel(), idx[1:, :].ravel(), idx[:-1, :].ravel()]
    vals = [(a_e + a_w + a_n + a_s).ravel(), -a_e[:, :-1].ravel(), -a_w[:, 1:].ravel(),
            -a_n[:-1, :].ravel(), -a_s[1:, :].ravel()]
    A = sparse.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                          shape=(n * n, n * n))
    return spsolve(A, np.full(n * n, f * h * h)).reshape(n, n)


def sample_split(n: int, seed: int, grid: int, chunk_size: int) -> np.ndarray:
    """(n, grid, grid) float32 a(x); chunk c uses default_rng([seed, c])."""
    return np.concatenate([
        sample_permeability(min(chunk_size, n - start), grid, np.random.default_rng([seed, c]))
        for c, start in enumerate(range(0, n, chunk_size))]).astype(np.float32)


def solve_block(job: tuple[str, int, np.ndarray]) -> tuple[str, int, np.ndarray]:
    """job = (split, start, a_block) -> (split, start, u_block float32)."""
    split, start, a = job
    return split, start, np.stack([solve(ai.astype(np.float64)) for ai in a]).astype(np.float32)


def parse_splits(spec: str) -> dict[str, tuple[int, int]]:
    """'train:9000:0,val:500:1' -> {'train': (9000, 0), 'val': (500, 1)}"""
    out = {}
    for item in spec.split(","):
        name, n, seed = item.split(":")
        out[name] = (int(n), int(seed))
    return out


def make_jobs(a_by_split: dict[str, np.ndarray], block: int) -> list[tuple[str, int, np.ndarray]]:
    return [(name, start, a[start:start + block])
            for name, a in a_by_split.items() for start in range(0, len(a), block)]


def fmt_time(seconds: float) -> str:
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h}h{m:02d}m{s:02d}s" if h else f"{m}m{s:02d}s"


def estimate(grid: int, n_total: int, workers: int, per_worker: int = 3) -> float:
    """Time a probe of per_worker samples on every worker at once (so contention between
    parallel solves is included); return the estimated wall time for n_total samples."""
    workers = max(1, workers)
    a = sample_permeability(workers * per_worker, grid, np.random.default_rng(12345))
    jobs = make_jobs({"probe": a.astype(np.float32)}, per_worker)
    if workers > 1:
        with Pool(workers) as pool:
            pool.map(fmt_time, range(workers))  # start the workers before timing
            t0 = time.time()
            pool.map(solve_block, jobs)
            dt = time.time() - t0
    else:
        t0 = time.time()
        list(map(solve_block, jobs))
        dt = time.time() - t0
    return dt / len(a) * n_total


def write_split(path: Path, a: np.ndarray, u: np.ndarray) -> None:
    xc = cell_centres(a.shape[-1]).astype(np.float32)
    with h5py.File(path, "w") as f:
        f.create_dataset("nu", data=a, compression="gzip", compression_opts=4)
        f.create_dataset("tensor", data=u[:, None], compression="gzip", compression_opts=4)
        f["x-coordinate"] = xc
        f["y-coordinate"] = xc
        f.attrs["beta"] = BETA


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--splits", default="train:9000:0,val:500:1,test:500:2")
    p.add_argument("--grid", type=int, default=128)
    p.add_argument("--workers", type=int, default=os.cpu_count() or 1)
    p.add_argument("--chunk-size", type=int, default=200, help="a(x) sampling batch (PDEBench: 200)")
    p.add_argument("--block", type=int, default=8, help="samples per solver job")
    p.add_argument("--out-dir", type=Path, default=Path("data"))
    args = p.parse_args()

    splits = parse_splits(args.splits)
    n_total = sum(n for n, _ in splits.values())
    args.out_dir.mkdir(parents=True, exist_ok=True)
    print(f"{n_total} samples at {args.grid}x{args.grid}, {args.workers} workers; "
          f"estimated {fmt_time(estimate(args.grid, n_total, args.workers))}", flush=True)

    t0 = time.time()
    a_by_split = {name: sample_split(n, seed, args.grid, args.chunk_size)
                  for name, (n, seed) in splits.items()}
    u_by_split = {name: np.empty_like(a) for name, a in a_by_split.items()}
    jobs = make_jobs(a_by_split, args.block)
    done = 0
    pool = Pool(args.workers) if args.workers > 1 else None
    results = pool.imap_unordered(solve_block, jobs) if pool else map(solve_block, jobs)
    last_print = 0.0
    try:
        for split, start, u in results:
            u_by_split[split][start:start + len(u)] = u
            done += len(u)
            if time.time() - last_print < 2.0 and done < n_total:
                continue  # print at most every 2 s, plus the final line
            last_print = time.time()
            elapsed = time.time() - t0
            rate = done / elapsed
            print(f"  {done:6d}/{n_total} ({100 * done / n_total:5.1f}%)  {rate:6.1f} samples/s  "
                  f"elapsed {fmt_time(elapsed)}  ETA {fmt_time((n_total - done) / rate)}", flush=True)
    finally:
        if pool:
            pool.close(); pool.join()

    meta = {"setting": "PDEBench 2D_DarcyFlow beta=1.0 inputs, exact steady solve",
            "grid": args.grid, "beta": BETA, "a_values": [LO, HI], "chunk_size": args.chunk_size,
            "workers": args.workers, "seconds": round(time.time() - t0, 1), "splits": {}}
    for name, (n, seed) in splits.items():
        a, u = a_by_split[name], u_by_split[name]
        out = args.out_dir / f"darcy_{name}.hdf5"
        write_split(out, a, u)
        meta["splits"][name] = {"n": n, "seed": seed, "file": out.name}
        print(f"wrote {out}  nu {a.shape}  tensor {(len(u), 1) + u.shape[1:]}", flush=True)
    (args.out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
    print(f"done in {fmt_time(time.time() - t0)}; wrote {args.out_dir / 'meta.json'}")


if __name__ == "__main__":
    main()

In [ ]:
# ETA: time a few samples on this machine and extrapolate to the full run.
import importlib, gen_darcy
importlib.reload(gen_darcy)
N_TOTAL = sum(n for n, _ in SPLITS.values())
eta = gen_darcy.estimate(GRID, N_TOTAL, N_WORKERS)
print(f"{N_TOTAL} samples at {GRID}x{GRID} on {N_WORKERS} workers: estimated {gen_darcy.fmt_time(eta)} "
      "(live progress + ETA is printed while the next cell runs)")

In [ ]:
splits_arg = ",".join(f"{name}:{n}:{seed}" for name, (n, seed) in SPLITS.items())
!python gen_darcy.py --splits {splits_arg} --grid {GRID} --workers {N_WORKERS} --chunk-size {CHUNK_SIZE} --out-dir {OUT_DIR}

In [ ]:
# Sanity checks: PDEBench layout, shapes, two-phase values, finiteness, disjoint splits.
import h5py, json
import numpy as np
data = {}
for s, (n, _) in SPLITS.items():
    with h5py.File(f"{OUT_DIR}/darcy_{s}.hdf5", "r") as f:
        assert {"nu", "tensor", "x-coordinate", "y-coordinate"} <= set(f), s
        assert f.attrs["beta"] == 1.0
        a, u = f["nu"][:], f["tensor"][:, 0]
    assert a.shape == u.shape == (n, GRID, GRID), (s, a.shape, u.shape)
    assert np.isfinite(a).all() and np.isfinite(u).all() and (u >= 0).all(), s
    assert set(np.unique(a)) <= {np.float32(0.1), np.float32(1.0)}, s
    data[s] = {"a": a, "u": u}
    print(f"{s:5s} nu {a.shape}  frac(a=1) {np.mean(a == 1):.3f}   "
          f"u min {u.min():.2e} max {u.max():.3f} mean {u.mean():.3f}")
hashes = {s: {x.tobytes() for x in d["a"]} for s, d in data.items()}
names = list(hashes)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        overlap = len(hashes[names[i]] & hashes[names[j]])
        assert overlap == 0, f"{names[i]}/{names[j]} share {overlap} samples"
print("splits are disjoint")
print(json.dumps(json.load(open(f"{OUT_DIR}/meta.json")), indent=1))

In [ ]:
# A few samples per split: permeability a(x) and solution u(x).
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for k, s in enumerate(SPLITS):
    for j in range(2):
        col = 2 * k + j
        axes[0, col].imshow(data[s]["a"][j].T, origin="lower", cmap="viridis")
        axes[0, col].set_title(f"{s} #{j}  a(x)")
        im = axes[1, col].imshow(data[s]["u"][j].T, origin="lower", cmap="magma")
        axes[1, col].set_title("u(x)")
        fig.colorbar(im, ax=axes[1, col], fraction=0.046, pad=0.04)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/samples.png", dpi=110)
plt.show()

**Next:** *Save Version* (Save & Run All). In `02_train_ddp.ipynb` use **Add Input → Your Work →
this notebook**; it auto-detects `darcy_train.hdf5` under `/kaggle/input`. To train on PDEBench
instead, attach `2D_DarcyFlow_beta1.0_Train.hdf5` and set `DATA_SOURCE = "pdebench"` there.